# Reproducing Luo, Jiang, Desai et al. (2024) from dandiset 001261

**Paper.** Luo F, Jiang L, Desai NS, Bai L, Watkins GV, Eldridge MAG, et al. (2024). Comparative
physiology and morphology of BLA-projecting NBM/SI cholinergic neurons in mouse and macaque.
*J. Comp. Neurol.* 532, e70001. [doi:10.1002/cne.70001](https://doi.org/10.1002/cne.70001). The
claims file was written from the version of record (open access, PMC11583843), with its supplement.

**Dandiset.** [DANDI:001261](https://dandiarchive.org/dandiset/001261/0.250917.2023), version
0.250917.2023. The paper cites no version (its Data Availability Statement gives only
https://dandiarchive.org/). This is the only published version, so it is pinned. The draft holds
the same 152 files (same checksums); it differs only in metadata (contributors, a related resource,
and the like), none of which the analysis uses. Files are read over HTTP into memory, not to disk.

**What the dandiset holds.** One NWB file per neuron: whole-cell current-clamp responses to a
family of current steps, at 10 kHz, from mice and macaques, with the group of the neuron (species,
cholinergic or not) in `session_description`. It holds no morphology, no quality-control measures
and no slice identifiers.

**Code.** The features follow the claims file, which takes them from Methods 2.5 and the paper's
MATLAB code (github.com/nsdesai/patch_clamp_analysis, commit 17a3265, doi:10.5281/zenodo.10975327),
re-implemented in Python in `nbmsi.py`; the UMAP is umap-learn in place of Meehan's MATLAB
implementation, and the LDA is scikit-learn's in place of MATLAB's fitcdiscr.

| Claim | Tested or blocked |
|---|---|
| sample-sizes | tested |
| fig2-mouse-c-nc-15-of-18 | tested |
| fig2d-mouse-max-firing-rate | tested |
| fig2d-mouse-threshold-rheobase | tested |
| fig2d-mouse-ahp | tested |
| fig2d-mouse-adaptation | tested |
| fig2d-mouse-passive | tested |
| figs2-2-mouse-amplitude-cv-sag | tested |
| fig2c-mouse-umap-separation | tested |
| fig5a-mouse-lda-electrophysiology | tested |
| fig5b-mouse-lda-ahp-amplitude-top | tested |
| fig7d-macaque-more-excitable | tested |
| fig7d-macaque-threshold-ahp | tested |
| fig7-macaque-spike-width | tested |
| figs7-1-macaque-amplitude-cv-sag-upstroke | tested |
| fig7-macaque-mouse-feature-count | tested |
| fig7d-macaque-input-resistance | tested |
| fig7c-macaque-mouse-umap-separation | tested |
| fig4d-mouse-morphology-3-of-13 | blocked, data missing (morphology) |
| figs4-1a-mouse-sholl-not-different | blocked, data missing (morphology) |
| fig4c-mouse-morphology-umap-similar | blocked, data missing (morphology) |
| fig5c-mouse-lda-morphology-fails | blocked, data missing (morphology) |
| fig9-macaque-convex-hull | blocked, data missing (morphology) |
| fig9-macaque-morphology-feature-count | blocked, data missing (morphology) |
| fig9-macaque-fewer-branch-points | blocked, data missing (morphology) |
| fig9c-morphology-umap-weak | blocked, data missing (morphology) |
| fig10-combined-separation | blocked, data missing (morphology) |
| fig10c-combined-lda-weights | blocked, data missing (morphology) |
| figs9-1a-macaque-mouse-sholl-similar | blocked, data missing (morphology) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import nbmsi

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 70)

started = time.time()
cells = nbmsi.load_all()
print(f"loaded {len(cells)} files from dandiset {nbmsi.DANDISET_ID} version {nbmsi.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
info = pd.DataFrame([{
    "i": i, "folder": c["path"].split("/")[0], "subject": c["subject"],
    "species": nbmsi.species_group(c["group"])[0], "grp": nbmsi.species_group(c["group"])[1],
    "sex": c["sex"], "age": c["age"], "identifier": c["identifier"].strip("'"), "n_sweeps": c["n_sweeps"],
    "sweep_s": c["sweep_s"], "rates": c["rates"], "units": c["units"], "bias": c["bias"], "hash": c["hash"],
    "notes": c["notes"]} for i, c in enumerate(cells)])
print("Files per group (session_description):")
print(info.groupby(["species", "grp"]).size().to_string())
print("\nSubjects per species:", info.groupby("species").subject.nunique().to_dict())
print("Subjects per group:", info.groupby(["species", "grp"]).subject.nunique().to_dict())
sexes = info.drop_duplicates("subject").groupby(["species", "sex"]).size()
print("Sex of subjects:", sexes.to_dict(), "(Table 1: mice 18 M, 9 F; macaques 7 M, 2 F)")
print("\nSampling rates:", sorted({r for x in info.rates for r in x}), "Hz;",
      "units (voltage, stimulus):", sorted({u for x in info.units for u in x}))
print("Amplifier fields, the same in every sweep of every file: bias_current", sorted({b for x in info.bias for b in x}), "A")
print("Sweeps per file:", info.n_sweeps.value_counts().sort_index().to_dict())
print("Sweep durations (s):", info.sweep_s.astype(str).value_counts().to_dict())
print("\nGroup labels (session_description):", info.assign(g=[c["group"] for c in cells]).g.value_counts().to_dict())
print("general/notes, the only free-text field per file; its values per species:")
print(info.groupby("species").notes.agg(lambda x: x.value_counts().to_dict()).to_string())


The dandiset holds 152 files, one neuron each, with the group of each neuron in
`session_description`: 48 mouse cholinergic, 46 mouse noncholinergic, 46 macaque cholinergic and 12
macaque noncholinergic files, from 27 mice and 9 macaques, as in the paper (Table 1 and Results
3.1). Nothing in the files records the quality-control measures (access resistance, offset
current), the slice, or the morphology. The amplifier fields (bias current 10 pA, and the
bridge balance and capacitance compensation, which are the same in every file) are the same in every sweep of every
file, although the neurons were held at -65 mV by a bias current set per neuron (Methods 2.4); they
are placeholders, not measurements. The one free-text field, `general/notes`, holds only a bregma
position (for example "Bregma (mm) -0.7"); no field records a slice or a quality-control measure.

In [ ]:
dups = info[info.duplicated("hash", keep=False)].sort_values("hash")
print("Files whose voltage traces are identical, sample for sample:")
print(dups[["i", "folder", "species", "grp", "identifier"]].to_string(index=False))
named = info.identifier.str.extract(r"(CG\d+)")[0]
other = info[named.notna() & (("sub-" + named) != info.folder)]
print("\nFiles whose identifier names another mouse than their folder:")
print(other[["i", "folder", "identifier"]].to_string(index=False))

Two recordings are deposited twice, with the same traces sample for sample. One mouse neuron,
`CG608_FL_S4C7_B+G+_-63mV.mat`, is filed once as cholinergic and once as noncholinergic; its name
says G+ (GFP-positive, which marks the cholinergic neurons in ChAT-tau-GFP mice), so it is counted
once, as cholinergic. One macaque noncholinergic neuron, `NHP_20220420-03`, is filed twice under
two session names. Each is counted once below: 48 mouse cholinergic, 45 mouse noncholinergic and
11 macaque noncholinergic neurons. Two files in the folder of mouse CG608 are named for CG609.

In [ ]:
pre, n = int(nbmsi.PRE_MS / nbmsi.DT), int(nbmsi.STEP_MS / nbmsi.DT)
rows = []
for i, c in enumerate(cells):
    for s in c["sweeps"]:
        rows.append({"i": i, "step_pA": s["step"], "duration_ms": s["duration_ms"], "onset_s": s["onset_s"],
                     "first_nonzero_s": s["first_nonzero_s"], "file_onset_s": c["onset_s"],
                     "pulse_to_step_ms": s["pulse_to_step_ms"], "test_pulse": s["test_pulse"],
                     "baseline_mV": s["baseline"]})
sw = pd.DataFrame(rows)
print("Stimulus values: the files label them amperes; the values are", sorted(float(x) for x in sw.step_pA.unique()))
print("Step durations (ms) and how many sweeps:", sw.duration_ms.value_counts(dropna=False).sort_index().to_dict())
print("Short pulses (< 200 ms): amplitudes", sorted(float(x) for x in sw[sw.test_pulse].step_pA.unique()), "in",
      sw[sw.test_pulse].i.nunique(), "files")
print("Files with steps above 200 pA:", sw[sw.step_pA > 200].i.nunique())
steps = sw[~sw.test_pulse].groupby("i").step_pA.apply(lambda x: sorted(x.unique()))
incr = steps.apply(lambda s: sorted({float(b - a) for a, b in zip(s, s[1:]) if 0 <= a < b <= 200}))
print("Increments between positive steps up to 200 pA (pA), by file:", incr.astype(str).value_counts().to_dict())
has = steps.apply(lambda s: (-60.0 in s, -20.0 in s, 200.0 in s))
print("Files with a -60 pA, a -20 pA and a 200 pA step:", int(has.apply(all).sum()), "of", len(has))
base = sw.groupby("i").baseline_mV.median()
print("\nMedian potential 10 to 1 ms before the step, per file (mV): 5th, 50th, 95th percentiles",
      np.percentile(base, [5, 50, 95]).round(1), "; files outside -100 to 0 mV:", list(base[(base < -100) | (base > 0)].round(0).items()))
rel = pd.Series([x for r in sw.pulse_to_step_ms for x in r])
print("\nSweeps holding a step and a separate short pulse:", int((sw.pulse_to_step_ms.str.len() > 0).sum()),
      "; start of the pulse relative to the step onset (ms), and how many sweeps:", rel.round(0).value_counts().sort_index().to_dict())
steps_only = sw[~sw.test_pulse & sw.duration_ms.notna()]
print("Steps whose onset (start of the longest run of non-zero current) differs from the file's onset (largest step):",
      int(((steps_only.onset_s - steps_only.file_onset_s).abs() > 1e-9).sum()), "of", len(steps_only))


In [ ]:
# Files whose sweeps start with a test pulse before the step: the step onset against the first non-zero sample
early = sw[(sw.first_nonzero_s < sw.onset_s - 1e-9)]
print("Sweeps whose first non-zero stimulus sample comes before the step (a test pulse first):", len(early), "in",
      early.i.nunique(), "files:", [info.identifier[i] for i in sorted(early.i.unique())])
print(early.groupby("i").agg(sweeps=("step_pA", "size"), first_nonzero_s=("first_nonzero_s", "median"),
                             step_onset_s=("onset_s", "median"), file_onset_s=("file_onset_s", "median"),
                             step_ms=("duration_ms", "median")).assign(identifier=lambda d: info.identifier[d.index].values).to_string())
ex = next(i for i in sorted(early.i.unique()) if info.species[i] == "mouse")
raw = nbmsi.raw_sweeps(cells[ex]["path"])
top = max(raw, key=lambda r: r["i"].max())
t = np.arange(len(top["v"])) / 1e4
fig, ax = plt.subplots(2, 1, figsize=(7, 4), sharex=True)
ax[0].plot(t, top["v"], lw=0.5); ax[0].set_ylabel("V (mV)")
ax[1].plot(t, top["i"], lw=0.8); ax[1].set_ylabel("I (pA)"); ax[1].set_xlabel("time in sweep (s)")
c0 = cells[ex]["sweeps"][[r["name"] for r in raw].index(top["name"])]
for a in ax:
    a.axvspan(c0["onset_s"], c0["onset_s"] + 0.5, color="0.85")
ax[0].set_title(f"{info.identifier[ex]}, {top['name']}: the analysed 500 ms shaded")
plt.show()
print(f"plotted: voltage {top['v'].min():.1f} to {top['v'].max():.1f} mV, current {top['i'].min():.0f} to {top['i'].max():.0f} pA, all inside the axes;",
      f"spikes in the analysed window: {len(nbmsi.peaks(c0['v'][int(nbmsi.PRE_MS / nbmsi.DT):]))},",
      f"in the 500 ms from the first non-zero sample: {len(nbmsi.peaks(top['v'][int(c0['first_nonzero_s'] * 1e4): int(c0['first_nonzero_s'] * 1e4) + 5000]))}")


In 170 sweeps of 8 files (six mouse cholinergic neurons of November 2021 to February 2022 and two
macaque neurons of M#11) a 100 ms, -20 pA test pulse starts at 0.3 s and the step at 1.0 s. The step
onset is taken as the start of the longest run of non-zero current in each sweep, which puts the
analysed window on the step in these files, at the file's onset (the first sample of its largest
step) in every step of every file. In every other sweep the pulse starts 1.3 s or more after the
step onset, outside the analysed window. In the plotted 200 pA sweep the analysed window holds 3
spikes, while the 500 ms from the first non-zero sample hold none. In the first round of this
attempt the onset was the first non-zero sample, which put the window on the test pulse in these 8
files: they then seemed not to depolarise with positive steps and to fire no spike.

The stimulus values are in pA, not amperes as the files say. The steps run from -60 (or -100) to
200 pA, in some files to 300 pA, and the increments are 10 to 50 pA in most files, 5 to 20 pA in
others, and 100 pA in one, not the 10 pA increments of the Fig. S2-1 legend; the rheobase can only be
as fine as the steps. Steps last 500 or 1000 ms; the first 500 ms are analysed, as the code does.
Most files also hold a short -20 pA pulse (50 or 100 ms), in a sweep of its own or in the same sweep
as a step, left out as not part of the family of steps. The potential before the steps is near -65 mV, as Methods 2.4 says, in all files but one: the
macaque file `2023_03_06_0002 R+B+A2C1 RMP -78.abf` sits near -1557 "mV", 20 times too large (its
name gives -78 mV, and -1557 / 20 = -77.9). Its scale is wrong, and it is left out of the analysis.

In [ ]:
oc = pd.DataFrame([nbmsi.onset_contrast(c) for c in cells], columns=["at_onset", "50_ms_earlier"])
oc[["species", "grp"]] = info[["species", "grp"]]
print("Change of potential at the onset of the -60 pA step (mean 5 to 25 ms after minus 20 to 0 ms before, mV),")
print("and the same windows 50 ms earlier:")
print(oc.groupby(["species", "grp"]).agg(["median", "min", "max"]).round(2).to_string())
resp = pd.Series([nbmsi.responsiveness(c) for c in cells])
print("\nSteady-state potential on the largest step (<= 200 pA) minus that on the most negative step (mV);")
print("the 12 smallest:")
print(pd.DataFrame({"i": resp.nsmallest(12).index, "mV": resp.nsmallest(12).round(1).values,
                    "group": info.loc[resp.nsmallest(12).index, ["species", "grp"]].agg(" ".join, axis=1).values,
                    "identifier": info.loc[resp.nsmallest(12).index, "identifier"].values}).to_string(index=False))

The voltage and the stimulus of each sweep share one clock: at the onset of the -60 pA step the
potential falls in every file, while the same windows 50 ms earlier show no change (group medians
-0.07 to 0.14 mV). Every neuron depolarises with positive steps: the steady-state potential on the
largest step lies 17.0 mV or more above that on the most negative step.

In [ ]:
info["rmp_name"] = [nbmsi.rmp_from_name(c["identifier"]) for c in cells]
print("Resting potential written in the file name (the only record of F1), per group:")
print(info.groupby(["species", "grp"]).rmp_name.agg(["count", "size"]).rename(columns={"count": "with value", "size": "files"}).to_string())

F1, the resting potential "measured in the absence of a current injection" (Methods 2.5), is not in
the data: every sweep is recorded with the holding current. Some of the original file names, kept as
the NWB identifier, carry a value (for example "RMP -49.4mV"): 8 of 48 mouse cholinergic, 28 of 46
mouse noncholinergic, 23 of 46 macaque cholinergic and 4 of 12 macaque noncholinergic files. F1 is
therefore not computed for the paper's neurons, and the named values are shown only as a variant.

## Features

The 18 features of Methods 2.5, as the claims file defines them (F1 to F18; see `nbmsi.py` for each
choice and its source). The neurons analysed: every file, each duplicated recording once, without
the file whose scale is wrong.

In [ ]:
drop = set(info[info.duplicated("hash", keep="first")].i)
cg608 = info[info.identifier.str.contains("CG608_FL_S4C7")]
drop = (drop - set(cg608.i)) | set(cg608[cg608.grp == "NC"].i)  # the G+ copy is kept
drop |= set(base[(base < -100) | (base > 0)].index)
use = [i for i in range(len(cells)) if i not in drop]
print("left out:", sorted(drop), [info.identifier[i] for i in sorted(drop)])

def table(repeat):
    rows = []
    for i in use:
        r = nbmsi.features(cells[i], repeat=repeat)
        r.update(i=i, subject=info.subject[i], species=info.species[i], grp=info.grp[i])
        rows.append(r)
    return pd.DataFrame(rows).set_index("i")

t0 = time.time()
feat = table("first")
feat_last = table("last")
print(f"features computed in {time.time() - t0:.0f} s")
print(feat.groupby(["species", "grp"]).size().to_string())
F = nbmsi.FEATURES
print("\nMissing values per feature and group (no spike up to 200 pA leaves the spike features missing):")
print(feat.groupby(["species", "grp"])[F].agg(lambda x: int(x.isna().sum())).T.to_string())
print("\nRange of every feature over the neurons analysed:")
print(feat[F].describe().T[["count", "min", "50%", "max"]].round(3).to_string())
print("\nSag step used (pA), per group:", feat.groupby(["species", "grp"]).sag_step.agg(lambda x: x.value_counts(dropna=False).to_dict()).to_dict())
moved = feat[feat.sag_step != nbmsi.SAG_STEP]
for i in moved.index:
    one = nbmsi._step_traces(cells[i])[1]
    if nbmsi.SAG_STEP in one:
        v = one[nbmsi.SAG_STEP]["v"][pre: pre + n]
        flat = int(np.max(np.diff(np.flatnonzero(np.diff(np.concatenate([[1], np.diff(v) != 0, [1]]))))))
        print(f"  {info.identifier[i]} ({info.species[i]} {info.grp[i]}): -60 pA response minimum {v.min():.1f} mV, "
              f"longest run of identical samples {flat}; sag on the {moved.sag_step[i]:.0f} pA step")
    else:
        print(f"  {info.identifier[i]} ({info.species[i]} {info.grp[i]}): no -60 pA step, steps {sorted(one)}; sag on the {moved.sag_step[i]:.0f} pA step")

# Variant: the macaque file with the wrong scale, its voltage divided by 20, added to the macaque neurons
bad = sorted(set(base[(base < -100) | (base > 0)].index))
resc = []
for i in bad:
    c = dict(cells[i]); c["sweeps"] = [dict(s, v=s["v"] / 20.0) for s in cells[i]["sweeps"]]
    r = nbmsi.features(c); r.update(i=i, subject=info.subject[i], species=info.species[i], grp=info.grp[i])
    resc.append(r)
feat_resc = pd.concat([feat, pd.DataFrame(resc).set_index("i")])
print("\nVariant, the file with the wrong scale divided by 20:", {k: round(float(v), 3) for k, v in resc[0].items() if k in F})


Readings. The main reading of each feature is the Methods' (claims file, F1 to F18): input
resistance and tau on the -20 pA step, latency to the threshold crossing, downstroke reading (a)
(first peak to 20 ms after it), AHP on the rheobase step within 100 ms of threshold, adaptation and
CV on the 200 pA step (reading a); the AHP width with its crossings searched within the 100 ms
window of F12, as the claims file's F14 says ("Window as in F12"). The other readings carry a suffix:
`downstroke_code` (F11 b), `adaptation_most` and `cv_isi_most` (F17 and F18 b, the step with the most
spikes); `ahp_width_to_end` (F14 with the crossings searched to the end of the step); and the code's
variants `rin_code`, `tau_code`, `latency_peak`, `ahp_*_code`, `rate_200` (the 200 pA reading of F16)
and `max_rate_code` (the code's maximum over every step, including those above 200 pA).
Sag is measured on the -60 pA step, or the closest step below it, and on the next step up when the
response is saturated (claims file, F2, from sagpotential.m; saturated read as 1000 or more identical
consecutive samples); `sag_60`, the -60 pA step whatever its response, is a variant.
`feat_last` takes the last sweep of a repeated amplitude in place of the first, and `feat_resc` adds
the macaque file with the wrong scale, its voltage divided by 20.

In [ ]:
M = (feat.species == "mouse")
mC, mNC = M & (feat.grp == "C"), M & (feat.grp == "NC")
qC = (feat.species == "macaque") & (feat.grp == "C")
EXTRA = ["downstroke_code", "adaptation_most", "cv_isi_most", "rin_code", "tau_code", "latency_peak",
         "ahp_amp_code", "ahp_latency_code", "ahp_width_code", "rate_200", "max_rate_code", "ahp_width_to_end", "sag_60"]
mouse = nbmsi.ranksum_table(feat, mC, mNC, F + EXTRA).rename(columns=lambda c: c.replace("_a", "_C").replace("_b", "_NC"))
species = nbmsi.ranksum_table(feat, qC, mC, F + EXTRA).rename(columns=lambda c: c.replace("_a", "_macaque").replace("_b", "_mouse"))
mouse_last = nbmsi.ranksum_table(feat_last, mC, mNC, F + EXTRA)
species_last = nbmsi.ranksum_table(feat_last, qC, mC, F + EXTRA)
qCr = (feat_resc.species == "macaque") & (feat_resc.grp == "C")
mCr = (feat_resc.species == "mouse") & (feat_resc.grp == "C")
species_resc = nbmsi.ranksum_table(feat_resc, qCr, mCr, F + EXTRA)
def show(tab):
    """A rank-sum table with the p-values in scientific notation."""
    return tab.to_string(float_format=lambda x: f"{x:.4g}", formatters={"p": "{:.2e}".format})

rmp = info.set_index("i").loc[use, "rmp_name"]
feat["rmp_name"] = rmp
print("Mouse cholinergic (C) against noncholinergic (NC), rank-sum per feature:")
print(show(mouse))

## Claims

Each section quotes the claim; the cells that show the result carry the tag `claim:<id>`.

### sample-sizes

> Electrophysiology was analysed for 48 BLA-projecting cholinergic and 46 BLA-projecting
> noncholinergic NBM/SI neurons from 27 mice (56 slices) and 46 BLA-projecting cholinergic neurons
> from nine macaques (38 slices); morphology for 31 mouse cholinergic and 44 mouse noncholinergic
> neurons.

In [ ]:
files = info.groupby(["species", "grp"]).size().rename("files")
distinct = info.drop_duplicates("hash").copy()
distinct.loc[distinct.identifier.str.contains("CG608_FL_S4C7"), "grp"] = "C"
recs = distinct.groupby(["species", "grp"]).size().rename("distinct recordings")
analysed = feat.groupby(["species", "grp"]).size().rename("analysed")
print(pd.concat([files, recs, analysed], axis=1).to_string())
print("\nSubjects:", info.groupby("species").subject.nunique().to_dict(), "; with cholinergic neurons:",
      info[info.grp == "C"].groupby("species").subject.nunique().to_dict())
print("Sex of subjects:", sexes.to_dict())
print("Slices: not recorded in the files. Morphology: not in the dandiset.")
print("Group labels in session_description:", sorted(info.assign(g=[c["group"] for c in cells]).g.unique()),
      "; none marks neurons of unknown ChAT status, so the NC reading with them added (getindices.m) is the same count.")


**Result: partially reproduced.** The dandiset has 48 mouse cholinergic files, 46 mouse noncholinergic files and 46 macaque cholinergic files, from 27 mice and 9 macaques, with the sexes of Table 1 (mice 18 M and 9 F, macaques 7 M and 2 F). One of the 46 mouse noncholinergic files repeats a recording also filed as cholinergic, so the dandiset holds 45 distinct noncholinergic recordings, one fewer than the 46 of the paper. The four group labels in `session_description` include none for neurons of unknown ChAT status, so the second reading of the noncholinergic group gives the same count. The macaque cholinergic neurons come from 6 of the 9 macaques. Slices and morphology are not in the dandiset.

### fig2-mouse-c-nc-15-of-18

> In mouse, BLA-projecting cholinergic and noncholinergic NBM/SI neurons differ in 15 of the 18
> electrophysiological features; spike width, upstroke and downstroke do not differ.

The rank-sum table is printed under "Features" above.

In [ ]:
SETS = {"F11 a, F17/F18 a (200 pA)": ["downstroke", "adaptation", "cv_isi"],
        "F11 a, F17/F18 b (most spikes)": ["downstroke", "adaptation_most", "cv_isi_most"],
        "F11 b, F17/F18 a (200 pA)": ["downstroke_code", "adaptation", "cv_isi"],
        "F11 b, F17/F18 b (most spikes)": ["downstroke_code", "adaptation_most", "cv_isi_most"]}
BASE = ["sag", "rin", "tau", "rheobase", "threshold", "amplitude", "width", "latency", "upstroke",
        "ahp_amp", "ahp_latency", "ahp_width", "fi_slope", "max_rate"]

# The direction the paper states (Results 3.1 and 3.3, Figs. S2-2 and S7-1): +1 when the first group
# (mouse C; macaque) is larger, -1 when smaller. Features whose direction the paper does not give are left out.
DIR_MOUSE = {"max_rate": -1, "threshold": 1, "rheobase": 1, "ahp_amp": 1, "ahp_width": 1, "adaptation": 1,
             "adaptation_most": 1, "tau": 1, "amplitude": 1, "cv_isi": -1, "cv_isi_most": -1, "sag": -1}
DIR_SPECIES = {"max_rate": 1, "rheobase": -1, "fi_slope": 1, "threshold": -1, "ahp_amp": -1, "ahp_width": -1,
               "width": 1, "amplitude": 1, "cv_isi": 1, "cv_isi_most": 1, "sag": 1, "upstroke": 1, "rin": 1}

def count(tab, extra_sets=SETS, direction=DIR_MOUSE):
    rows = []
    a, b = [c for c in tab.columns if c.startswith("mean_")]
    for name, ex in extra_sets.items():
        cols = BASE + ex
        sig = tab.loc[cols, "p"] < 0.05
        wrong = [c for c in cols if sig[c] and c in direction
                 and np.sign(tab.loc[c, a] - tab.loc[c, b]) != direction[c]]
        rows.append({"readings": name, "features": len(cols), "differing": int(sig.sum()),
                     "not differing": ", ".join(c for c in cols if not sig[c]),
                     "differing against the paper's direction": ", ".join(wrong)})
    return pd.DataFrame(rows)

print("Mouse C against NC, features differing at p < 0.05, F1 not computable (17 features):")
print(count(mouse).to_string(index=False))
print("\nThe same with the last sweep of a repeated amplitude:")
print(count(mouse_last).to_string(index=False))
code_sets = {"code variants (rin, tau, latency, AHP of the code)":
             ["downstroke", "adaptation", "cv_isi"]}
alt = mouse.copy()
for a, b in [("rin", "rin_code"), ("tau", "tau_code"), ("latency", "latency_peak"), ("ahp_amp", "ahp_amp_code"),
             ("ahp_latency", "ahp_latency_code"), ("ahp_width", "ahp_width_code")]:
    alt.loc[a] = mouse.loc[b]
print("\nVariant, the code's choices for rin, tau, latency and the AHP (not deciding):")
print(count(alt, {"F11 a, F17/F18 a, code variants": SETS["F11 a, F17/F18 a (200 pA)"],
                  "F11 b, F17/F18 b, code variants": SETS["F11 b, F17/F18 b (most spikes)"]}).to_string(index=False))
r = nbmsi.ranksum_table(feat, mC, mNC, ["rmp_name"])
print("\nVariant, F1 from the file names (subset):", r.round(4).to_dict("records"))


**Result: reproduced.** F1 cannot be computed, so the count is out of 17 and the criterion's range is 13 to 15. With the adaptation index and the CV on the step with the most spikes (F17 and F18 reading b), 13 of the 17 features differ at p < 0.05 under either downstroke reading; with the 200 pA step (reading a), 12. Spike width, upstroke and downstroke are among the features that do not differ, as the criterion requires, together with AHP latency. One of the 13, the AHP width, differs in the direction opposite to the paper's (noncholinergic neurons have the wider AHP; see fig2d-mouse-ahp), so counting only differences in the paper's direction would give 12. The last sweep of a repeated amplitude gives the same counts, and the code's variants for input resistance, tau, latency and the AHP give 12 or 13, none against the paper's direction.

### fig2d-mouse-max-firing-rate

> Mouse noncholinergic BLA-projecting neurons are more excitable than cholinergic ones, with a
> maximum firing rate about six times larger (39.0 +- 24.9 against 7.3 +- 4.3 Hz).

In [ ]:
print(show(mouse.loc[['max_rate', 'rate_200', 'max_rate_code']]))

pa = feat[M].groupby(["grp", "subject"])[['max_rate']].agg(["count", "mean"]).round(2)
print("\nPer mouse (count of neurons, mean):")
print(pa.to_string())

fig, ax = plt.subplots(figsize=(4, 3))
for k, (m, lab) in enumerate([(mC, "C"), (mNC, "NC")]):
    v = feat.loc[m, "max_rate"]
    ax.plot(np.full(len(v), k) + np.random.default_rng(1).uniform(-0.15, 0.15, len(v)), v, "o", ms=3)
ax.set_xticks([0, 1], ["mouse C", "mouse NC"]); ax.set_ylabel("max firing rate (Hz)")
plt.show()
print("plotted range:", feat.loc[M, "max_rate"].min(), "to", feat.loc[M, "max_rate"].max(), "Hz; all points inside the axes")

**Result: partially reproduced.** Noncholinergic neurons fire faster (rank-sum p = 1.9e-12): mean maximum rate 6.92 +- 4.23 Hz for cholinergic and 34.8 +- 23.1 Hz for noncholinergic neurons, against 7.3 +- 4.3 and 39.0 +- 24.9 in the paper. The cholinergic mean and both SDs are within tolerance; the noncholinergic mean is just below its tolerance (35.1 to 42.9 Hz). The 200 pA reading gives 6.81 and 34.47 Hz.

### fig2d-mouse-threshold-rheobase

> Mouse cholinergic BLA-projecting neurons have a more depolarized spike threshold (-33 +-
> 4.6 against -37.1 +- 6.3 mV) and a larger rheobase (64.3 +- 41.2 against 42.2 +- 33.2 pA)
> than noncholinergic ones.

In [ ]:
print(show(mouse.loc[['threshold', 'rheobase']]))

pa = feat[M].groupby(["grp", "subject"])[['threshold', 'rheobase']].agg(["count", "mean"]).round(2)
print("\nPer mouse (count of neurons, mean):")
print(pa.to_string())

print("\nRheobase values (pA) by group:", {g: feat.loc[m, "rheobase"].value_counts().sort_index().to_dict() for g, m in [("C", mC), ("NC", mNC)]})

**Result: partially reproduced.** All four means are within tolerance: threshold -33.12 mV (cholinergic) and -36.77 mV (noncholinergic), rheobase 62.45 and 42.05 pA. The threshold test holds (p = 6.5e-4, cholinergic more depolarized). The rheobase is larger in cholinergic neurons, but p = 1.1e-3, just above the paper's 0.001. The SDs are 4.55 and 6.02 mV (paper 4.6 and 6.3) and 36.3 and 34.1 pA (paper 41.2 and 33.2). The rheobase can only take the step values of each file, which are 5 to 50 pA apart. One cholinergic and one noncholinergic neuron fire no spike up to 200 pA and have no rheobase.

### fig2d-mouse-ahp

> Mouse cholinergic BLA-projecting neurons have a larger AHP (28.9 +- 8.3 against 12.4 +- 6.1
> mV) and a wider AHP (177.9 +- 89.3 against 85.0 +- 118.6 ms) than noncholinergic ones.

In [ ]:
print(show(mouse.loc[['ahp_amp', 'ahp_width', 'ahp_width_to_end', 'ahp_amp_code', 'ahp_width_code']]))

pa = feat[M].groupby(["grp", "subject"])[['ahp_amp', 'ahp_width']].agg(["count", "mean"]).round(2)
print("\nPer mouse (count of neurons, mean):")
print(pa.to_string())

print("\nLast sweep of a repeated amplitude:")
print(show(mouse_last.loc[["ahp_amp", "ahp_width", "ahp_amp_code", "ahp_width_code"]]))

**Result: not reproduced.** With the Methods' window (rheobase step, 100 ms after threshold) the AHP amplitude is larger in cholinergic neurons (31.26 against 16.35 mV, p = 2.5e-13), with the cholinergic mean within tolerance and the noncholinergic mean outside it (11.16 to 13.64 mV). The AHP width, with its crossings searched within the 100 ms window as F14 says, is longer in noncholinergic neurons (298.9 against 183.1 ms, p = 4.1e-5), the opposite of the paper; with the crossings searched to the end of the step it is 226.3 against 160.0 ms (p = 0.017), the same way. The width test does not hold, so the criterion's tests do not both hold. The code's window (the first step with two spikes) gives 30.06 and 12.14 mV and 184.3 and 77.4 ms, all four within tolerance, with p = 4.1e-14 and 5.2e-8: the paper's values match the code's window, not the Methods'. Whether the claims file should judge the AHP on the code's window is asked of a reader.

### fig2d-mouse-adaptation

> Mouse cholinergic BLA-projecting neurons have a higher adaptation index than noncholinergic
> ones (0.83 +- 0.56 against 0.64 +- 0.29).

In [ ]:
print(show(mouse.loc[['adaptation', 'adaptation_most']]))

print("\nLast sweep of a repeated amplitude:")
print(show(mouse_last.loc[["adaptation", "adaptation_most"]]))

**Result: reproduced.** On the step with the most spikes (reading b) the adaptation index is 0.875 +- 0.586 for cholinergic and 0.606 +- 0.296 for noncholinergic neurons (paper 0.83 and 0.64, both within tolerance), p = 0.011. On the 200 pA step (reading a) it is 0.617 against 0.553, p = 0.86, which does not hold.

### fig2d-mouse-passive

> Mouse cholinergic BLA-projecting neurons have a more depolarized resting potential (-54.4
> +- 8.4 against -58.7 +- 8.2 mV) and a longer membrane time constant (34.6 +- 14.3 against
> 22.3 +- 9.8 ms) than noncholinergic ones.

In [ ]:
print(show(mouse.loc[['tau', 'tau_code']]))

print("\nF1 from the file names (subset; not the paper's neurons as a whole):")
print(show(nbmsi.ranksum_table(feat, mC, mNC, ["rmp_name"])))

pa = feat[M].groupby(["grp", "subject"])[['tau']].agg(["count", "mean"]).round(2)
print("\nPer mouse (count of neurons, mean):")
print(pa.to_string())

# Quality of the tau fits: fraction of the variance of the trace (2 to 500 ms of the -20 pA step) the fit explains
print("\nFit R^2 of tau, per group: median, minimum; neurons with R^2 < 0.9:")
print(feat.groupby(["species", "grp"]).tau_r2.agg(["median", "min", lambda x: int((x < 0.9).sum())]).round(3).to_string())
out = feat[M & (feat.tau > 100)].sort_values("tau", ascending=False)
print("\nMouse neurons with tau above 100 ms:")
print(out[["grp", "subject", "tau", "tau_r2", "tau_code", "rin"]].assign(identifier=info.identifier[out.index].values).round(3).to_string())
pre_ = int(nbmsi.PRE_MS / nbmsi.DT)
show_i = list(out.index[:3])
if show_i:
    fig, ax = plt.subplots(1, len(show_i), figsize=(3.2 * len(show_i), 2.6), squeeze=False)
    for a, i in zip(ax[0], show_i):
        s20 = nbmsi._step_traces(cells[i])[1][-20.0]
        v = s20["v"]; tt = (np.arange(len(v)) - pre_) * nbmsi.DT
        a.plot(tt, v, lw=0.6)
        a.set_title(f"{info.identifier[i][:18]}\ntau {feat.tau[i]:.0f} ms, R2 {feat.tau_r2[i]:.2f}", fontsize=8)
        a.set_xlabel("ms from step onset")
        print(f"{info.identifier[i]}: -20 pA trace {v.min():.1f} to {v.max():.1f} mV, inside the axes")
    ax[0][0].set_ylabel("V (mV)")
    plt.show()


**Result: not reproduced.** The resting potential is not in the data, so the claim is judged on tau. Tau on the -20 pA step is longer in cholinergic neurons (39.92 against 28.59 ms), but p = 3.2e-3, not below the paper's 0.0001, and both means are above their tolerance (31.14 to 38.06 and 20.07 to 24.53 ms), so tau does not hold. The code's step (the most negative unsaturated step from -50 to 0 pA) gives 34.88 against 22.04 ms, both within tolerance, p = 2.7e-6. The fits on the -20 pA step explain a median 84% (cholinergic) and 91% (noncholinergic) of the variance of the trace, and 33 cholinergic and 21 noncholinergic fits explain less than 90%; the two mouse neurons with tau above 100 ms (153.9 and 116.6 ms) have R^2 of 0.83 and 0.84. For the 8 cholinergic and 27 noncholinergic neurons whose file name gives a resting potential, it is -54.72 against -58.53 mV, close to the paper's values, but not different (p = 0.36).

### figs2-2-mouse-amplitude-cv-sag

> In mouse, spike amplitude, the CV of interspike intervals and sag also differ between
> cholinergic and noncholinergic BLA-projecting neurons, with cholinergic neurons having
> larger spikes, a lower CV and a smaller sag.

In [ ]:
print(show(mouse.loc[['amplitude', 'cv_isi', 'cv_isi_most', 'sag', 'sag_60']]))

print("\nLast sweep of a repeated amplitude:")
print(show(mouse_last.loc[["cv_isi", "cv_isi_most"]]))

**Result: reproduced.** Spike amplitude is larger in cholinergic neurons (59.65 against 53.35 mV, p = 5.4e-5, within the figure's p <= 0.001). The CV of the interspike intervals is lower in cholinergic neurons on the step with the most spikes (0.190 against 0.317, p = 1.3e-3, within p <= 0.01); on the 200 pA step p = 0.011. Sag is smaller in cholinergic neurons (1.86 against 3.19 mV, p = 1.8e-3, within p <= 0.01). The sag rests on the claims file's rule for a saturated -60 pA response: in two cholinergic and five noncholinergic mouse neurons that response is clipped at -100 or -105.5 mV for 1144 to 4793 samples, and the next step up is used. On the -60 pA step whatever its response the sag is 1.80 against 3.01 mV, p = 0.011.

### fig2c-mouse-umap-separation

> A UMAP of the 18 electrophysiological features separates mouse cholinergic from
> noncholinergic BLA-projecting neurons almost completely, with only two neurons on the wrong
> side of the boundary.

UMAP procedure U of the claims file on the 17 computable features (Methods readings), 10 seeds, 1000
label permutations per seed, under both readings of missing values.

In [ ]:
U17 = ["sag", "rin", "tau", "rheobase", "threshold", "amplitude", "width", "latency", "upstroke", "downstroke", "ahp_amp", "ahp_latency", "ahp_width", "fi_slope", "max_rate", "adaptation", "cv_isi"]
def umap_runs(mask, ylab, seed):
    rng = np.random.default_rng(seed)
    X = feat.loc[mask, U17].to_numpy(float)
    y = (feat.loc[mask, "grp"] if ylab == "grp" else feat.loc[mask, "species"]).to_numpy()
    out = {}
    for miss in ["drop", "impute"]:
        out[miss] = nbmsi.umap_separation(X, y, miss, range(10), 1000, rng)
    return out

t0 = time.time()
u_mouse = umap_runs(M, "grp", 20)
for miss, r in u_mouse.items():
    print(f"missing values: {miss}; neurons {r.n[0]}, PCs kept {r.pcs[0]}")
    print(r.round(4).to_string(index=False))
    print("median misclassified over seeds:", r.misclassified.median(), "range", r.misclassified.min(), "to",
          r.misclassified.max(), "; seeds beating every permutation:", int((r.misclassified < r.null_min).sum()), "of 10")
print(f"{time.time() - t0:.0f} s")

# Sensitivity to the UMAP parameters (umap-learn), Methods readings, missing values dropped and imputed; 200 permutations
sens = []
for nn_, md_ in [(5, 0.3), (30, 0.3), (15, 0.1), (15, 0.5)]:
    for miss in ["drop", "impute"]:
        r = nbmsi.umap_separation(feat.loc[M, U17].to_numpy(float), feat.loc[M, "grp"].to_numpy(), miss, range(10), 200,
                                  np.random.default_rng(21), n_neighbors=nn_, min_dist=md_)
        sens.append({"n_neighbors": nn_, "min_dist": md_, "missing": miss, "median misclassified": r.misclassified.median(),
                     "range": f"{r.misclassified.min()} to {r.misclassified.max()}",
                     "seeds beating every permutation": int((r.misclassified < r.null_min).sum())})
print("\nSensitivity to the UMAP parameters:")
print(pd.DataFrame(sens).to_string(index=False))


**Result: not reproduced.** The UMAP separates the groups better than every one of the 1000 label permutations for all 10 seeds, under both readings of missing values, but a linear boundary in the UMAP plane misclassifies a median of 4 neurons (range 4 to 6, of 85) when neurons with missing values are dropped and 8.5 (8 to 10, of 93) when they are imputed, not the paper's 2. With n_neighbors 5 or 30, or min_dist 0.1 or 0.5, the median is 4 to 7 (dropped) and 8 to 10.5 (imputed), never 2 or fewer.

### fig5a-mouse-lda-electrophysiology

> A linear discriminant analysis of the electrophysiological features clearly separates mouse
> cholinergic from noncholinergic BLA-projecting neurons.

LDA procedure L: resubstitution accuracy against 1000 shuffles of each kind, both scalings and both
readings of missing values; 10-fold cross-validated accuracy as a variant.

In [ ]:
def lda_runs(mask, ylab, seed):
    rng = np.random.default_rng(seed)
    X = feat.loc[mask, U17].to_numpy(float)
    y = (feat.loc[mask, "grp"] if ylab == "grp" else feat.loc[mask, "species"]).to_numpy()
    return pd.DataFrame([nbmsi.lda_separation(X, y, miss, scale, 1000, rng)
                         for miss in ["drop", "impute"] for scale in ["z", "raw"]])

l_mouse = lda_runs(M, "grp", 50)
print(l_mouse.round(4).to_string(index=False))

**Result: reproduced.** The LDA classifies 98.8% of the 85 neurons with every feature (missing values dropped) and 94.6% of all 93 (missing values imputed), for both scalings, above the 95th percentile (0.74 to 0.76) and the maximum (0.81 to 0.84) of 1000 shuffles of either kind (p = 0.001). The 10-fold cross-validated accuracy, with the imputation and the scaling fitted on the training folds only, is 0.89 to 0.95.

### fig5b-mouse-lda-ahp-amplitude-top

> AHP amplitude has the highest weight in the LDA separating mouse cholinergic from
> noncholinergic BLA-projecting neurons.

In [ ]:
for miss in ["drop", "impute"]:
    w = nbmsi.lda_weights(feat.loc[M, U17].to_numpy(float), feat.loc[M, "grp"].to_numpy(), U17, miss)
    for col in w.columns:
        top = w[col].sort_values(ascending=False).head(5)
        print(f"missing {miss}, weights {col}: top five", top.round(3).to_dict())

**Result: reproduced.** AHP amplitude has the largest weight under both readings of the weights and both readings of missing values (absolute coefficient 9.50 and 6.62, next 7.49 and 1.91; coefficient times mean difference 14.28 and 9.79, next 9.22 and 2.49).

### fig7d-macaque-more-excitable

> Macaque BLA-projecting cholinergic neurons are more excitable than mouse ones, with a
> maximum firing rate three times larger (26.3 +- 23.1 against 7.3 +- 4.3 Hz), a rheobase a
> third as large (23.3 +- 16.5 against 64.3 +- 41.2 pA) and an f-I slope twice as large (0.13
> +- 0.10 against 0.06 +- 0.03 Hz/pA).

In [ ]:
print(show(species.loc[['max_rate', 'rheobase', 'fi_slope', 'rate_200', 'max_rate_code']]))

pq = feat[feat.grp == "C"].groupby(["species", "subject"])[["max_rate", "rheobase", "fi_slope"]].agg(["count", "mean"]).round(3)
print("\nPer animal, cholinergic neurons (count, mean):")
print(pq.to_string())
print("\nVariant (not deciding), the macaque file with the wrong scale divided by 20 and included:")
print(show(species_resc.loc[['max_rate', 'rheobase', 'fi_slope']]))


**Result: partially reproduced.** All three tests hold in the reported direction at p < 0.0001: maximum rate 21.47 against 6.92 Hz (p = 7.2e-12), rheobase 22.33 against 62.45 pA (p = 6.1e-11), f-I slope 0.122 against 0.058 Hz/pA (p = 9.3e-6). The rheobase and both f-I slopes are within tolerance; the macaque maximum rate, 21.47 Hz, is below the paper's 26.3 (tolerance 23.67 to 28.93). The code's maximum over every step, including steps above 200 pA, gives 24.04 Hz, and with the rescaled macaque file added the maximum rate is 22.3 Hz; these variants do not decide.

### fig7d-macaque-threshold-ahp

> Macaque BLA-projecting cholinergic neurons have a more hyperpolarized spike threshold
> (-39.7 +- 5.9 against -33.0 +- 4.6 mV), a smaller AHP (19.1 +- 8.8 against 28.9 +- 8.3 mV)
> and a narrower AHP (106.6 +- 71.9 against 177.9 +- 89.3 ms) than mouse ones.

In [ ]:
print(show(species.loc[['threshold', 'ahp_amp', 'ahp_width', 'ahp_width_to_end', 'ahp_amp_code', 'ahp_width_code']]))

print("\nLast sweep of a repeated amplitude:")
print(show(species_last.loc[["ahp_amp", "ahp_width", "ahp_amp_code", "ahp_width_code"]]))
pq = feat[feat.grp == "C"].groupby(["species", "subject"])[["threshold", "ahp_amp", "ahp_width"]].agg(["count", "mean"]).round(2)
print("\nPer animal, cholinergic neurons (count, mean):")
print(pq.to_string())
print("\nVariant (not deciding), the macaque file with the wrong scale divided by 20 and included:")
print(show(species_resc.loc[['threshold', 'ahp_amp', 'ahp_width']]))


**Result: partially reproduced.** Two of the three tests hold: the macaque threshold is more hyperpolarized (-39.42 against -33.12 mV, p = 2.4e-7) and the AHP smaller (20.76 against 31.26 mV, p = 6.8e-9), both macaque means within tolerance. With the Methods' window the AHP width does not differ (224.4 against 183.1 ms, p = 0.17, macaque longer; 187.0 against 160.0 ms, p = 0.32, with the crossings searched to the end of the step). The code's window gives 119.1 against 184.3 ms (p = 3.5e-4), the macaque narrower, as in the paper, its mean just above the tolerance (95.9 to 117.3 ms).

### fig7-macaque-spike-width

> Macaque BLA-projecting cholinergic neurons have wider spikes than mouse ones (1.32 +- 0.45
> against 1.15 +- 0.43 ms).

In [ ]:
print(show(species.loc[['width']]))
print("\nVariant (not deciding), the macaque file with the wrong scale divided by 20 and included:")
print(show(species_resc.loc[['width']]))


**Result: not reproduced.** Spikes are wider in macaque (1.314 against 1.157 ms), both means within tolerance, but p = 0.013, not below the paper's 0.01, so the criterion's test does not hold. With the rescaled macaque file added, p = 0.016.

### figs7-1-macaque-amplitude-cv-sag-upstroke

> Macaque BLA-projecting cholinergic neurons also differ from mouse ones in spike amplitude,
> CV of interspike intervals, sag and upstroke, each larger in macaque, while the downstroke
> does not differ.

In [ ]:
print(show(species.loc[['amplitude', 'cv_isi', 'cv_isi_most', 'sag', 'sag_60', 'upstroke', 'downstroke', 'downstroke_code']]))

print("\nLast sweep of a repeated amplitude:")
print(show(species_last.loc[["cv_isi", "cv_isi_most"]]))
print("\nVariant (not deciding), the macaque file with the wrong scale divided by 20 and included:")
print(show(species_resc.loc[['amplitude', 'cv_isi', 'cv_isi_most', 'sag', 'upstroke', 'downstroke', 'downstroke_code']]))


**Result: partially reproduced.** Three of the four differences hold in the stated direction at the figure's level: spike amplitude (65.31 against 59.65 mV, p = 1.8e-3), CV of the intervals (0.451 against 0.183 on the 200 pA step, p = 1.1e-5; 0.494 against 0.190 on the step with the most spikes, p = 3.1e-7) and upstroke (170.1 against 120.5 mV/ms, p = 1.1e-5). Sag does not differ (3.64 against 1.86 mV, p = 0.090; on the -60 pA step whatever its response, p = 0.066). The downstroke, which the claim says is shared, differs under both readings (-54.38 against -65.17 mV/ms, p = 7.6e-3), which also keeps the claim at most partially reproduced.

### fig7-macaque-mouse-feature-count

> Most of the 18 electrophysiological features differ between macaque and mouse
> BLA-projecting cholinergic neurons; the text gives the number as 15 of 18, and the figure
> legends imply 17.

In [ ]:
print("Macaque C against mouse C, rank-sum per feature:")
print(show(species))
print("\nFeatures differing at p < 0.05 (17 features, F1 not computable):")
print(count(species, direction=DIR_SPECIES).to_string(index=False))
print("\nLast sweep of a repeated amplitude:")
print(count(species_last, direction=DIR_SPECIES).to_string(index=False))
print("\nVariant (not deciding), the macaque file with the wrong scale divided by 20 and included:")
print(count(species_resc, direction=DIR_SPECIES).to_string(index=False))
alt = species.copy()
for a_, b_ in [("rin", "rin_code"), ("tau", "tau_code"), ("latency", "latency_peak"), ("ahp_amp", "ahp_amp_code"),
               ("ahp_latency", "ahp_latency_code"), ("ahp_width", "ahp_width_code")]:
    alt.loc[a_] = species.loc[b_]
print("\nVariant, the code's choices for rin, tau, latency and the AHP (not deciding):")
print(count(alt, {"F11 a, F17/F18 a, code variants": SETS["F11 a, F17/F18 a (200 pA)"],
                  "F11 b, F17/F18 b, code variants": SETS["F11 b, F17/F18 b (most spikes)"]}, DIR_SPECIES).to_string(index=False))

**Result: reproduced.** With F1 not computable, 14 of the 17 features differ between macaque and mouse cholinergic neurons with the adaptation index and the CV on the step with the most spikes (either downstroke reading), and 13 with the 200 pA step. The criterion's range for the reading of 15 is 14 to 16, so the count of 14 is at its lower edge; the criterion does not lower the range when F1 is missing. Sag, AHP latency and AHP width do not differ under any set of readings, and none of the differing features goes against the paper's direction. The last sweep of a repeated amplitude, the rescaled macaque file and the code's variants give 13 or 14. The resting potential from the file names (22 macaque and 8 mouse neurons) does not differ (p = 0.98), so counting F1 would not raise the count.

### fig7d-macaque-input-resistance

> Macaque BLA-projecting cholinergic neurons have a larger input resistance and a smaller
> rheobase than mouse ones, the opposite of what larger cells would show.

In [ ]:
print(show(species.loc[['rin', 'rin_code', 'rheobase']]))
print("\nVariant (not deciding), the macaque file with the wrong scale divided by 20 and included:")
print(show(species_resc.loc[['rin', 'rheobase']]))


**Result: reproduced.** On the -20 pA step the macaque input resistance is larger (593.2 against 355.5 MOhm, p = 2.9e-5) and the rheobase smaller (22.33 against 62.45 pA, p = 6.1e-11). The code's slope over the steps from -50 to 0 pA gives 394.4 against 335.1 MOhm, p = 0.37, which does not hold.

### fig7c-macaque-mouse-umap-separation

> A UMAP of the 18 electrophysiological features shows a strong separation between macaque
> and mouse BLA-projecting cholinergic neurons.

In [ ]:
u_species = umap_runs(feat.grp == "C", "species", 30)
for miss, r in u_species.items():
    print(f"missing values: {miss}; neurons {r.n[0]}, PCs kept {r.pcs[0]}")
    print(r.round(4).to_string(index=False))
    print("median misclassified over seeds:", r.misclassified.median(), "range", r.misclassified.min(), "to",
          r.misclassified.max(), "; seeds beating every permutation:", int((r.misclassified < r.null_min).sum()), "of 10")
sens = []
for nn_, md_ in [(5, 0.3), (30, 0.3), (15, 0.1), (15, 0.5)]:
    for miss in ["drop", "impute"]:
        r = nbmsi.umap_separation(feat.loc[feat.grp == "C", U17].to_numpy(float), feat.loc[feat.grp == "C", "species"].to_numpy(),
                                  miss, range(10), 200, np.random.default_rng(31), n_neighbors=nn_, min_dist=md_)
        sens.append({"n_neighbors": nn_, "min_dist": md_, "missing": miss, "median misclassified": r.misclassified.median(),
                     "range": f"{r.misclassified.min()} to {r.misclassified.max()}",
                     "seeds beating every permutation": int((r.misclassified < r.null_min).sum())})
print("\nSensitivity to the UMAP parameters:")
print(pd.DataFrame(sens).to_string(index=False))


**Result: blocked, methods underspecified.** The separation is better than every one of the 1000 label permutations for all 10 seeds under both readings of missing values (reading b holds), but a linear boundary misclassifies a median of 9 (range 9 to 10, of 88) and 9 (8 to 10, of 93) of the neurons, not at most 2 (reading a fails). With n_neighbors 5 or 30, or min_dist 0.1 or 0.5, the median is 8.5 to 10.5. The criterion makes this case methods_underspecified: the paper does not say how strong a separation it calls strong.

### Blocked claims

The eleven claims on morphology (Figs. 4, 5C, 9, 10, S4-1 and S9-1) need the reconstructions'
features and Sholl counts. The dandiset holds only current-clamp recordings (its asset summary lists
CurrentClampSeries and CurrentClampStimulusSeries, and no file holds a morphology table or image),
and the paper's Data Availability Statement names only the DANDI Archive. They are blocked,
`data_missing`.

## Summary

| Claim | Outcome |
|---|---|
| sample-sizes | partially reproduced |
| fig2-mouse-c-nc-15-of-18 | reproduced |
| fig2d-mouse-max-firing-rate | partially reproduced |
| fig2d-mouse-threshold-rheobase | partially reproduced |
| fig2d-mouse-ahp | not reproduced |
| fig2d-mouse-adaptation | reproduced |
| fig2d-mouse-passive | not reproduced |
| figs2-2-mouse-amplitude-cv-sag | reproduced |
| fig2c-mouse-umap-separation | not reproduced |
| fig5a-mouse-lda-electrophysiology | reproduced |
| fig5b-mouse-lda-ahp-amplitude-top | reproduced |
| fig7d-macaque-more-excitable | partially reproduced |
| fig7d-macaque-threshold-ahp | partially reproduced |
| fig7-macaque-spike-width | not reproduced |
| figs7-1-macaque-amplitude-cv-sag-upstroke | partially reproduced |
| fig7-macaque-mouse-feature-count | reproduced |
| fig7d-macaque-input-resistance | reproduced |
| fig7c-macaque-mouse-umap-separation | blocked, methods underspecified |
| Eleven morphology claims (Figs. 4, 5C, 9, 10, S4-1, S9-1) | blocked, data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The features are re-implemented in Python from the claims file's definitions (Methods 2.5 and the paper's MATLAB code); the UMAP is umap-learn and the LDA scikit-learn's. The resting potential (F1) is not in the data, so the counts are out of 17 features. Where the Methods and the code differ, the Methods decide; the code's windows for the AHP and for tau match the paper's values much better than the Methods'. Steps above 200 pA and the short test pulses are left out, and the step onset is read from each sweep's longest run of current. Each duplicated recording is counted once, and one macaque file with a wrong scale is left out (shown rescaled as a variant). The paper's quality control cannot be repeated, as its measures are not in the files.

What was found about the dataset. The group labels in `session_description` give the paper's counts, but one mouse recording is deposited twice, once as cholinergic and once as noncholinergic, and one macaque recording twice. One macaque file is scaled 20 times too large. In 8 files a test pulse precedes the step in every sweep, while in the others it follows the step. The -60 pA responses of 8 neurons are clipped at -100 or -105.5 mV. The stimulus is labelled amperes but holds pA. The amplifier fields (bias current, bridge balance, capacitance compensation) hold one value in every file. The step amplitudes differ between files and are mostly not the 10 pA increments of the Fig. S2-1 legend. The resting potential is not recorded, except in some file names. Two files in the folder of mouse CG608 are named for CG609. The morphology is not in the dandiset.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")